# Development Script

Jan 14th, 2026

Maxime Bouthillier

### University of Waterloo - MMATH CM Research Project

In [1]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import random
import re
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from typing import List
from dotenv import load_dotenv
import os
from torch import Tensor

In [2]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [4]:
# Loading Note Events dataset
df = pd.read_csv("/work/mbouthil/projects/physionet.org/files/mimiciii/1.4/NOTEEVENTS.csv.gz", nrows=1000)
df = df[['SUBJECT_ID', 'TEXT']]
notes = df['TEXT'].tolist()

# Exploring Notes:

## Note Splitting

### Chunking based on headers

In [188]:
note = df['TEXT'].iloc[3]

def detail_merge(sections:list[str], min_words:int=10):
    merged = ['Details:']

    for section in sections:
        word_count = len(section.split())

        if word_count < min_words:
            merged[0] = merged[0] + "\n" + section
        
        else:
            merged.append(section)

    return merged


def header_split(note: str, min_words:int=10) -> list[str]:
    pattern = r'(?:^|\n)([A-Za-z][A-Za-z ]{3,50}):\s*(.*?)(?=(?:\n[A-Za-z][A-Za-z ]{3,50}:)|$)'
    matches = re.findall(pattern, note, flags=re.DOTALL)

    if not matches:
        return [note]

    # Reconstruct header + content
    sections = [f"{h}: {c}".strip() for h, c in matches if c.strip()]
    sections = [" ".join([word for word in section.split(' ') if word]) for section in sections]

    sections = detail_merge(sections, min_words)
    

    return sections

note = header_split(note)

In [87]:
def paragraph_splits(note:str) -> list[str]:

    paragraphs = re.split(r"\n\s*\n\n+", note)
    return [p.strip() for p in paragraphs if p.strip()]

In [95]:
splits = paragraph_splits(note)

for i in splits:
    print(i)
    print('END')

Admission Date:  [**2119-5-4**]              Discharge Date:   [**2119-5-25**]
END
Service: CARDIOTHORACIC

Allergies:
Amlodipine

Attending:[**Last Name (NamePattern1) 1561**]
Chief Complaint:
81 yo F smoker w/ COPD, severe TBM, s/p tracheobronchoplasty [**5-5**]
s/p perc trach [**5-13**]

Major Surgical or Invasive Procedure:
bronchoscopy 3/31,4/2,3,[**6-12**], [**5-17**], [**5-19**]
s/p trachealplasty [**5-5**]
percutaneous tracheostomy [**5-13**] after failed extubation
down size trach on [**5-25**] to size 6 cuffless
END
History of Present Illness:
This 81 year old woman has a history of COPD. Over the past five

years she has had progressive difficulties with her breathing.
In
[**2118-6-4**] she was admitted to [**Hospital1 18**] for respiratory failure
due
to a COPD exacerbation. Due to persistent hypoxemia, she
required
intubation and a eventual bronchoscopy on [**2118-6-9**] revealed marked

narrowing of the airways on expiration consistent with
tracheomalacia.
She subsequentl

### Chunking regarless of content:

In [ ]:
def chunk_by_sentence(text: str, max_chars: int=400) -> list[str]:
    
    """
    Split text into chunks of at most max_chars characters,
    preserving sentence boundaries.

    Parameters
    ----------
    text : str
        Input text to be chunked.
    max_chars : int
        Target maximum character length per chunk.

    Returns
    -------
    List[str]
        List of text chunks.
    """

    # 1. Split text into sentences (keeps punctuation)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # Edge case: single sentence longer than max_chars
        if len(sentence) > max_chars:
            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""
            chunks.append(sentence.strip())
            continue

        # Try to append sentence to current chunk
        if len(current_chunk) + len(sentence) + 1 <= max_chars:
            current_chunk += (" " if current_chunk else "") + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [225]:
splits = chunk_by_sentence(df['TEXT'].iloc[3], 400)

In [226]:
print(len(splits))

45


# Additional Synthetic Query Generation

In [24]:
# Loading Query Passage Dataset
df = pd.read_csv('/work/mbouthil/projects/research_project/MEDRAG/processed_data/synq.csv')
df = df.drop(columns=["TEXT", "NOTE"])

In [25]:
queries = ['Subject ID: ' + str(df['SUBJECT_ID'].iloc[i]) + '\n'  + str(df['QUERY'].iloc[i]) for i in range(len(df))]
passages = ['Subject ID: ' + str(df['SUBJECT_ID'].iloc[i]) + '\n'  + str(df['PASSAGE'].iloc[i]) for i in range(len(df))]

In [83]:
query_test = [query[18:] for query in queries[:2]]
print(query_test)

['Does the patient have a history of tuberculosis prior to this admission?', 'Has the patient had any previous imaging studies or medical evaluations for osteoporosis prior to this current assessmen']


In [14]:
system_prompt = '''
You are a helpful AI Assistant. You will be provided a question written by a doctor.

Your task is to rewrite this question 7 different ways. However, it is crucial that these alternative questions ask the same
underlying question as the provided question.
Feel free to use more or less medical terminology and medical acronyms where you see fit. 
Moreover, keep the questions relatively simple and straight forward. 

Format your output as follows:

**Query_1**

**Query_2**

...

**Query_7** 
'''

In [ ]:
def llm_batch(notes: list[str], system_prompt:str) -> list[str]:
    messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note}
        ]
        for note in notes
    ]

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_p=0.9,
            do_sample=True
        )

    responses = []
    for i in range(len(notes)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

def chunked(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]

In [84]:
add_queries = llm_1(query_test, system_prompt)

In [85]:
def queries_split(queries:list[str]) -> list[list[str]]:

    queries = [re.split(r"\*\*Query_\d+\*\*\s*", query)[1:] for query in queries]
    queries = [[query.replace('/n', '').strip() for query in query_list if query.replace('/n', '').strip()] for query_list in queries]
    return queries

split_queries = queries_split(add_queries)

In [86]:
def df_expansion(df:pd.DataFrame, add_queries:list[list[str]]) -> pd.DataFrame:

    new_df = pd.DataFrame(columns=df.columns)

    for i in range(len(df)):
        queries = add_queries[i]
        row = df.iloc[[i]]

        duplicates = pd.concat([row] * len(queries), ignore_index=True)
        duplicates.loc[:, "QUERY"] = queries

        new_rows = pd.concat([row, duplicates], ignore_index=True)
        new_df = pd.concat([new_df, new_rows], ignore_index=True)

    return new_df

In [87]:
test_df = pd.concat([df.iloc[[0]], df.iloc[[1]]], ignore_index=True)
test_df

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,QUERY,PASSAGE
0,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Does the patient have a history of tuberculosi...,Admission Date: [**2151-7-16**] Dischar...
1,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Has the patient had any previous imaging studi...,HEAD CT: Head CT showed no intracranial hemor...


In [88]:
test_df = df_expansion(test_df, split_queries)

In [89]:
test_df

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,QUERY,PASSAGE
0,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Does the patient have a history of tuberculosi...,Admission Date: [**2151-7-16**] Dischar...
1,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Does the patient have a past medical history o...,Admission Date: [**2151-7-16**] Dischar...
2,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Has the patient ever been diagnosed with tuber...,Admission Date: [**2151-7-16**] Dischar...
3,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Is there a history of tuberculosis in the pati...,Admission Date: [**2151-7-16**] Dischar...
4,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Was tuberculosis a pre-existing condition for ...,Admission Date: [**2151-7-16**] Dischar...
5,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Does the patient have a known history of tuber...,Admission Date: [**2151-7-16**] Dischar...
6,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Is tuberculosis a part of the patient's past m...,Admission Date: [**2151-7-16**] Dischar...
7,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Has the patient previously been treated for tu...,Admission Date: [**2151-7-16**] Dischar...
8,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Has the patient had any previous imaging studi...,HEAD CT: Head CT showed no intracranial hemor...
9,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Has the patient undergone any prior imaging st...,HEAD CT: Head CT showed no intracranial hemor...


### Testing outputs

In [7]:
test_df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/add_synq.csv")
test_df.head(10)

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,NOTE,QUERY,PASSAGE
0,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a history of tuberculosi...,Admission Date: [**2151-7-16**] Dischar...
1,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a past medical history o...,Admission Date: [**2151-7-16**] Dischar...
2,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Is there any prior history of tuberculosis in ...,Admission Date: [**2151-7-16**] Dischar...
3,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Has the patient ever been diagnosed with tuber...,Admission Date: [**2151-7-16**] Dischar...
4,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Is tuberculosis a part of the patient's past m...,Admission Date: [**2151-7-16**] Dischar...
5,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a known history of tuber...,Admission Date: [**2151-7-16**] Dischar...
6,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Is there any documented history of tuberculosi...,Admission Date: [**2151-7-16**] Dischar...
7,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Has the patient ever had a diagnosis of tuberc...,Admission Date: [**2151-7-16**] Dischar...
8,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,HEAD CT: Head CT showed no intracranial hemor...,Has the patient had any previous imaging studi...,HEAD CT: Head CT showed no intracranial hemor...
9,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,HEAD CT: Head CT showed no intracranial hemor...,Has the patient undergone any prior imaging st...,HEAD CT: Head CT showed no intracranial hemor...


# Testing the trained Retreiver

In [3]:
df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/processed_data/synq.csv")
passages = ['Subject ID: ' + str(df['SUBJECT_ID'].iloc[i]) + '\n'  + str(df['PASSAGE'].iloc[i]) for i in range(len(df))]
queries = ['Subject ID: ' + str(df['SUBJECT_ID'].iloc[i]) + '\n'  + str(df['QUERY'].iloc[i]) for i in range(len(df))]

In [100]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
passage_encoder = AutoModel.from_pretrained("bert-base-uncased")

passage_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/MEDRAG/model_weights/passage_encoder"
) # .to("cuda")

passage_encoder.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
def encode_passage(notes:list[str], batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        for i in range(0, len(notes), batch_size):
            batch = notes[i:i+batch_size]

            inputs = tokenizer(
                batch, 
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=512
            ) #.to("cuda")

        outputs = passage_encoder(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0] 

        embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)


In [119]:
test_passages = passages
embeddings = encode_passage(test_passages)

KeyboardInterrupt: 

In [7]:
len(synq)

28431

# Actual Retrieval

In [20]:
import faiss 
import json

In [21]:
# Loading embeddings and confirming shape
embeddings_np = np.load("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_embeddings.npy")
embeddings_np.shape

(28431, 768)

In [22]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage.index")

# Load metadata
metadata = []
with open("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

In [23]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained("bert-base-uncased")

query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/MEDRAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ) #.to("cuda")

    outputs = query_encoder(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0] 

    embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)

In [28]:
print(queries[0])
print("/n/n")
print(passages[0])

Subject ID: 22532
Does the patient have a history of tuberculosis prior to this admission?
/n/n
Subject ID: 22532
Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]


Service:
ADDENDUM:

RADIOLOGIC STUDIES:  Radiologic studies also included a chest
CT, which confirmed cavitary lesions in the left lung apex
consistent with infectious process/tuberculosis. This also
moderate-sized left pleural effusion.


In [25]:
query = "Subject ID: 22532\nHas the patient had teberculosis?"
query_emb = encode_query([query]).detach().cpu().numpy()

In [26]:
K = 20
scores, ids = index.search(query_emb, K)

In [31]:
print(scores)

[[278.67456 278.33157 277.29712 276.59564 275.72748 275.01962 274.72366
  274.43604 274.25043 274.155   274.0747  273.98962 273.84848 273.5051
  272.88623 272.64194 272.63574 272.4034  272.3411  272.23718]]


In [37]:
candidates = [metadata[i]["text"] for i in ids[0]]
for i in candidates:
    if "22532" in i:
        print(i)
    print("\n\n")

## Skipping Re-ranker

In [169]:
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [170]:
system_prompt = '''
You are a helpful AI Assistant. 

Use the following provided context to asnwer the question. 
If the answer is not contained within the context, only return "No information found".
'''

note = f'''
Context: {candidates[0]}
Question: {query}

Answer:
'''

messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": note}]

In [175]:
prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        top_p=0.9,
        do_sample=True
    )

input_len = inputs["input_ids"].shape[1]
gen_tokens = outputs[0, input_len:]

response = tokenizer.decode(gen_tokens, skip_special_tokens=True)

In [178]:
print(response[11:].replace("/n", ''))

Yes, the patient has been confirmed to have cavitary lesions in the left lung apex consistent with an infectious process/tuberculosis.


# Additinoal Synthetic Query Trouble Shooting

In [5]:
# Loading in Synthetic Query dataset
df = pd.read_csv('/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/add_synq.csv')

In [7]:
len(df)

800